# 4. Prepare the data for the transformer

1. **Splits.** Every R3 track and the 12 cells with the most tracks in each condition go to the test set
   ("low train / high test"). The remaining tracks are split 90/10 into train and validation. Whole tracks
   stay in one split. Settings: `splits:` in `configs/pipeline.yaml`.
2. **What the model sees.** Each window becomes 20 tokens of 3 frames: (sum dx, sum dy, mean heading).
3. **Augmentation.** Contrastive pretraining compares each window with a shuffled, rescaled, rotated copy.
4. **Augmentation screen** (supplementary figure): how often each augmentation would change the gate label.

In [ ]:
import sys
sys.path.insert(0, "..")  # so `import sptnano` works from this folder

import polars as pl

from sptnano import config

paths = config.load_paths()
settings = config.load_settings()

import matplotlib.pyplot as plt
import numpy as np
import torch

from sptnano import augment, splits, tokens

## 1. Splits

Writes `train_df.parquet`, `val_df.parquet`, `test_df.parquet` (frame rows) and `data_splits.pkl` to a new folder.
Same as `python scripts/make_splits.py --out_dir <folder>`.

In [ ]:
RUN_SPLITS = False  # True to make a new split; the paper split is paths["splits_dir"]

if RUN_SPLITS:
    new_splits_dir = paths["splits_dir"].parent / "data_splits_certain_lowtrain_hightest_new"
    splits.make_splits(
        paths["gated_dir"] / "instant_df_hierarchical_gates.parquet",
        paths["gated_dir"] / "windowed_df_hierarchical_gates.parquet",
        new_splits_dir,
        settings["splits"],
    )
    splits.check_splits(new_splits_dir)
else:
    splits.check_splits(paths["splits_dir"])

## 2. One window as tokens, and one augmented copy

In [ ]:
val_frames = pl.read_parquet(paths["splits_dir"] / "val_df.parquet", n_rows=5000)
dataset = tokens.WindowDataset(val_frames, show_progress=False)
sample = dataset.samples[0]
n_tokens = sample["n_tokens"]
augment_tokens = augment.make_augmentation(settings["transformer"]["augmentation_segment_tokens"])

window_tokens = torch.from_numpy(sample["tokens"]).unsqueeze(0)
augmented_tokens = augment_tokens(window_tokens, torch.tensor([n_tokens]))[0].numpy()

fig, axes = plt.subplots(1, 3, figsize=(11, 3.2))
for ax, (title, values) in zip(axes[:2], [("tokens", sample["tokens"]), ("augmented tokens", augmented_tokens)]):
    ax.imshow(values[:n_tokens].T, aspect="auto", cmap="coolwarm", interpolation="nearest")
    ax.set_yticks([0, 1, 2], ["sum dx", "sum dy", "heading"])
    ax.set_xlabel("token (3 frames)")
    ax.set_title(title)
for values, label in [(sample["tokens"], "original"), (augmented_tokens, "augmented")]:
    axes[2].plot(np.cumsum(values[:n_tokens, 0]), np.cumsum(values[:n_tokens, 1]), label=label)
axes[2].set_aspect("equal")
axes[2].set_title("path (um)")
axes[2].legend(frameon=False)
fig.tight_layout()

## 3. Augmentation screen (supplementary figure)

Ported unchanged from `AUGMENTATION_SCAN_MINIMAL_FIGURE_2026.ipynb`. Each augmented window gets its gate
features recomputed and is re-gated; pink panels changed category. Saves to `paper_plots_dir/plots`.

In [ ]:
# Imports and configuration
from __future__ import annotations

import os
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.patches import Circle, Polygon, Rectangle
from matplotlib.path import Path as MplPath
import numpy as np
import pandas as pd
import polars as pl
import seaborn as sns
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from tqdm.notebook import tqdm

sys.path.insert(0, '..')
from sptnano import config
from sptnano.augment_screen import compute_features, get_window_frames, has_complete_frames

paths = config.load_paths()

DATA_DIR = paths['gated_dir']
INSTANT_DF_PATH = DATA_DIR / 'instant_df_hierarchical_gates.parquet'
WINDOWED_DF_PATH = DATA_DIR / 'windowed_df_hierarchical_gates.parquet'
SOURCE_LABEL_COL = 'final_population_certain'

OUTPUT_DIR = paths['paper_plots_dir'] / 'plots'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
rng = np.random.default_rng(SEED)

# Match the report notebook's four behavior-state labels.
GATE_STATE_ORDER = ['bound', 'confined', 'transport', 'exploratory']
GATE_STATE_COLORWAY_NAME = 'Dark2_discrete_matplotlib'
_dark2 = plt.cm.get_cmap('Dark2', len(GATE_STATE_ORDER))
GATE_STATE_COLOR_LIST = list(_dark2(np.linspace(0, 1, len(GATE_STATE_ORDER))))
GATE_STATE_COLORS = dict(zip(GATE_STATE_ORDER, GATE_STATE_COLOR_LIST))

POP_COLORS = {
    **GATE_STATE_COLORS,
    'mixed_exploratory_bound': '#BDBDBD',
    'ungatedgate1': '#9E9E9E',
    'ungatedgate2': '#9E9E9E',
    'unknown': '#9E9E9E',
}

FINAL_POP_ORDER = ['bound', 'confined', 'transport', 'exploratory']
POP_LABELS = {
    'bound': 'Bound',
    'confined': 'SubDiff',
    'transport': 'SuperDiff',
    'exploratory': 'Exploratory',
    'mixed_exploratory_bound': 'Mixed',
    'ungatedgate1': 'Ungated',
    'ungatedgate2': 'Ungated',
    'unknown': 'Unknown',
}

GATE1_RECTS = {
    'low_speed_low_intersections': {'x': (0.0, 0.2), 'y': (0.0, 0.14), 'color': '#D0D0D0'},
    'high_speed_low_intersections': {'x': (0.2, 1.0), 'y': (0.0, 0.14), 'color': POP_COLORS['exploratory']},
    'low_speed_high_intersections': {'x': (0.0, 0.3), 'y': (0.14, 1.0), 'color': POP_COLORS['bound']},
    'high_speed_high_intersections': {'x': (0.3, 1.0), 'y': (0.14, 1.0), 'color': POP_COLORS['mixed_exploratory_bound']},
}
GATE2_BOUNDS = {
    'superdiffusive': [(1.1, 0.0), (2.825, 30.0), (3.0, 30.0), (3.0, 0.0)],
    'subdiffusive': [(-0.5, 0.0), (-0.5, 30.0), (2.825, 30.0), (1.1, 0.0)],
}

print('Output:', OUTPUT_DIR)
print('Palette:', GATE_STATE_COLORWAY_NAME)
for state in GATE_STATE_ORDER:
    print(f'  {POP_LABELS[state]:>12}: {POP_COLORS[state]}')

In [ ]:
# Load source tables and prepare original gated universe.
# Start from the same "certain" gated dataframes used by the transformer behavior report.
instant_df = pl.read_parquet(INSTANT_DF_PATH)
windowed_df = pl.read_parquet(WINDOWED_DF_PATH)

if SOURCE_LABEL_COL not in windowed_df.columns:
    raise KeyError(f"Missing source label column {SOURCE_LABEL_COL!r}; available columns include: {windowed_df.columns[:30]}")

if 'bad_fit_flag' in windowed_df.columns:
    windowed_df = windowed_df.filter(pl.col('bad_fit_flag') == False)
windowed_df = windowed_df.filter(pl.col(SOURCE_LABEL_COL).is_in(FINAL_POP_ORDER))

valid_uids = windowed_df['window_uid'].unique().to_list()
instant_df = instant_df.filter(pl.col('window_uid').is_in(valid_uids))

# Fit Gate 1 scaler on the same source data, preserving feature-space geometry for augmented windows.
fx, fy = 'avg_speed_um_s', 'self_intersections'
scaler = Pipeline([('std', StandardScaler()), ('mm', MinMaxScaler())])
scaler.fit(windowed_df[[fx, fy]].to_pandas())

print('Source:', DATA_DIR)
print('Source label column:', SOURCE_LABEL_COL)
print('Windowed:', windowed_df.shape)
print('Instant:', instant_df.shape)
display(windowed_df.group_by(SOURCE_LABEL_COL).agg(pl.len().alias('n')).sort('n', descending=True))

In [ ]:
# Augmentation specifications
SHUFFLE_TITRATION = [
    {'name': 'recalculated', 'label': 'Recalc\ncontrol', 'nseg': None, 'scale': None, 'angle': (0, 0)},
    {'name': 'shuffle_2seg', 'label': 'Shuffle\n2 seg', 'nseg': 2, 'scale': None, 'angle': (0, 0)},
    {'name': 'shuffle_3seg', 'label': 'Shuffle\n3 seg', 'nseg': 3, 'scale': None, 'angle': (0, 0)},
    {'name': 'shuffle_4seg', 'label': 'Shuffle\n4 seg', 'nseg': 4, 'scale': None, 'angle': (0, 0)},
    {'name': 'shuffle_6seg', 'label': 'Shuffle\n6 seg', 'nseg': 6, 'scale': None, 'angle': (0, 0)},
]

SCALE_TITRATION = [
    {'name': 'recalculated', 'label': 'Recalc\ncontrol', 'nseg': None, 'scale': None, 'angle': (0, 0)},
    {'name': 'shuffle_3seg', 'label': 'Shuffle\nonly', 'nseg': 3, 'scale': None, 'angle': (0, 0)},
    {'name': 'shuffle_3seg_scale_10', 'label': '3 seg +\nscale 0-10%', 'nseg': 3, 'scale': (0.90, 1.10), 'angle': (0, 0)},
    {'name': 'shuffle_3seg_scale_25', 'label': '3 seg +\nscale 0-25%', 'nseg': 3, 'scale': (0.75, 1.25), 'angle': (0, 0)},
    {'name': 'shuffle_3seg_scale_50', 'label': '3 seg +\nscale 10-50%', 'nseg': 3, 'scale': (0.50, 1.50), 'angle': (0, 0)},
]

ANGLE_TITRATION = [
    {'name': 'recalculated', 'label': 'Recalc\ncontrol', 'nseg': None, 'scale': None, 'angle': (0, 0)},
    {'name': 'shuf_3s_s10_50_a0', 'label': '3 seg +\nscale only', 'nseg': 3, 'scale': (0.50, 1.50), 'angle': (0, 0)},
    {'name': 'shuf_3s_s10_50_a1_5', 'label': '+ 1-5 deg\nangle', 'nseg': 3, 'scale': (0.50, 1.50), 'angle': (1, 5)},
    {'name': 'shuf_3s_s10_50_a5_10', 'label': '+ 5-10 deg\nangle', 'nseg': 3, 'scale': (0.50, 1.50), 'angle': (5, 10)},
    {'name': 'shuf_3s_s10_50_a20_30', 'label': '+ 20-30 deg\nangle', 'nseg': 3, 'scale': (0.50, 1.50), 'angle': (20, 30)},
    {'name': 'shuf_3s_s10_50_a50_60', 'label': '+ 50-60 deg\nangle', 'nseg': 3, 'scale': (0.50, 1.50), 'angle': (50, 60)},
    {'name': 'shuf_3s_s10_50_a80_90', 'label': '+ 80-90 deg\nangle', 'nseg': 3, 'scale': (0.50, 1.50), 'angle': (80, 90)},
]

PANEL_SPECS = {
    'A_shuffle': SHUFFLE_TITRATION,
    'B_scale': SCALE_TITRATION,
    'C_angle': ANGLE_TITRATION,
}

In [ ]:
# Helper functions for augmentation and re-gating
def get_original_xy_dt(uid):
    tdf = get_window_frames(instant_df, uid)
    if tdf.height == 0:
        return None, None, None
    return tdf['x_um'].to_numpy(), tdf['y_um'].to_numpy(), tdf['delta_time_s'].to_numpy()


def bidirectional_scale_factor(scale_range, random_state):
    if scale_range is None:
        return 1.0
    lo, hi = scale_range
    if np.isclose(lo, 1.0) and np.isclose(hi, 1.0):
        return 1.0
    if random_state.random() < 0.5:
        upper = min(0.95, hi)
        return random_state.uniform(lo, upper) if lo < upper else lo
    lower = max(1.05, lo)
    return random_state.uniform(lower, hi) if lower < hi else hi


def apply_titration_aug(x, y, spec, seed):
    if spec['name'] == 'recalculated':
        return x.copy(), y.copy()
    random_state = np.random.default_rng(seed)
    n = len(x)
    dx = np.diff(x, prepend=x[0])
    dy = np.diff(y, prepend=y[0])
    nseg = int(spec.get('nseg') or 1)
    idx_segments = np.array_split(np.arange(n), nseg)
    segs = []
    for idx in idx_segments:
        scale = bidirectional_scale_factor(spec.get('scale'), random_state)
        segs.append((dx[idx].copy() * scale, dy[idx].copy() * scale))
    order = np.arange(len(segs))
    random_state.shuffle(order)
    new_dx = np.concatenate([segs[i][0] for i in order])
    new_dy = np.concatenate([segs[i][1] for i in order])
    min_ang, max_ang = spec.get('angle') or (0, 0)
    if max_ang > 0:
        r = np.sqrt(new_dx**2 + new_dy**2)
        theta = np.arctan2(new_dy, new_dx)
        magnitudes = random_state.uniform(np.deg2rad(min_ang), np.deg2rad(max_ang), size=len(theta))
        signs = random_state.choice([-1, 1], size=len(theta))
        theta = theta + magnitudes * signs
        new_dx = r * np.cos(theta)
        new_dy = r * np.sin(theta)
    return np.cumsum(new_dx), np.cumsum(new_dy)


def gate1_name_from_features(avg_speed_um_s, self_intersections):
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        sx, sy = scaler.transform([[avg_speed_um_s, self_intersections]])[0]
    for gate_name, params in GATE1_RECTS.items():
        (xlo, xhi), (ylo, yhi) = params['x'], params['y']
        if xlo <= sx <= xhi and ylo <= sy <= yhi:
            return gate_name
    return 'ungated'


def gate2_name_from_features(alpha, cum_disp):
    point = (alpha, cum_disp)
    if MplPath(GATE2_BOUNDS['superdiffusive']).contains_point(point):
        return 'high_a_low_disp'
    if MplPath(GATE2_BOUNDS['subdiffusive']).contains_point(point):
        return 'low_a_high_disp'
    return 'ungated'


def final_population_from_features(features):
    gate1 = gate1_name_from_features(features['avg_speed_um_s'], features['self_intersections'])
    if gate1 == 'low_speed_low_intersections':
        gate2 = gate2_name_from_features(features['anomalous_exponent'], features['cum_displacement_um'])
        if gate2 == 'high_a_low_disp':
            return 'transport'
        if gate2 == 'low_a_high_disp':
            return 'confined'
        return 'ungatedgate2'
    if gate1 == 'high_speed_low_intersections':
        return 'exploratory'
    if gate1 == 'low_speed_high_intersections':
        return 'bound'
    if gate1 == 'high_speed_high_intersections':
        return 'mixed_exploratory_bound'
    return 'ungatedgate1'


def augmented_record(uid, spec):
    x, y, dt = get_original_xy_dt(uid)
    if x is None:
        return None
    seed = hash(f'{uid}_{spec["name"]}') % 2**32
    xa, ya = apply_titration_aug(x, y, spec, seed)
    features = compute_features(xa, ya, dt)
    pop = final_population_from_features(features)
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        g1_xy = scaler.transform([[features['avg_speed_um_s'], features['self_intersections']]])[0]
    return {
        'x': xa - xa[0],
        'y': ya - ya[0],
        'features': features,
        'final_population': pop,
        'g1_x': float(g1_xy[0]),
        'g1_y': float(g1_xy[1]),
        'g2_x': float(features['anomalous_exponent']),
        'g2_y': float(features['cum_displacement_um']),
    }

In [ ]:
# Select five complete examples per included final_population_certain category.
N_EXAMPLES_PER_POP = 5

def sample_complete_uids(pop, n_needed):
    all_uids = windowed_df.filter(pl.col(SOURCE_LABEL_COL) == pop)['window_uid'].to_list()
    rng.shuffle(all_uids)
    complete = []
    for uid in all_uids:
        if has_complete_frames(instant_df, uid):
            complete.append(uid)
            if len(complete) >= n_needed:
                break
    return complete

selected_for_fig = {pop: sample_complete_uids(pop, N_EXAMPLES_PER_POP) for pop in FINAL_POP_ORDER}
for pop, uids in selected_for_fig.items():
    print(f'{POP_LABELS[pop]:>12}: {len(uids)}')

In [ ]:
# Plot final panels: trajectories + Gate 1 + Gate 2 + heatmap
def draw_gate1_background(ax):
    for _gate, params in GATE1_RECTS.items():
        ax.add_patch(Rectangle(
            (params['x'][0], params['y'][0]),
            params['x'][1] - params['x'][0],
            params['y'][1] - params['y'][0],
            alpha=0.10,
            fc=params['color'],
            ec=params['color'],
            lw=0.4,
        ))
    ax.set_xlim(-0.04, 1.04)
    ax.set_ylim(-0.04, 1.04)
    ax.set_aspect('equal')
    ax.set_xticks([])
    ax.set_yticks([])


def draw_gate2_background(ax):
    ax.add_patch(Polygon(GATE2_BOUNDS['superdiffusive'], alpha=0.10, fc=POP_COLORS['transport'], ec=POP_COLORS['transport'], lw=0.4))
    ax.add_patch(Polygon(GATE2_BOUNDS['subdiffusive'], alpha=0.10, fc=POP_COLORS['confined'], ec=POP_COLORS['confined'], lw=0.4))
    ax.set_xlim(0, 3)
    ax.set_ylim(0, 30)
    ax.set_xticks([])
    ax.set_yticks([])


def plot_titration_panel(panel_specs, panel_title, out_stem):
    n_rows = sum(len(v) for v in selected_for_fig.values())
    n_traj_cols = len(panel_specs)
    n_cols = 1 + n_traj_cols + 3
    fig, axes = plt.subplots(
        n_rows,
        n_cols,
        figsize=(1.18 * n_cols, 1.02 * n_rows),
        gridspec_kw={'width_ratios': [0.36] + [1] * n_traj_cols + [1.05, 1.05, 0.72]},
    )
    if n_rows == 1:
        axes = axes.reshape(1, -1)

    record_cache = {}
    pop_limits = {}
    for pop, uids in selected_for_fig.items():
        xmin = ymin = np.inf
        xmax = ymax = -np.inf
        for uid in uids:
            for spec in panel_specs:
                rec = augmented_record(uid, spec)
                if rec is None:
                    continue
                record_cache[(uid, spec['name'])] = rec
                xmin, xmax = min(xmin, np.nanmin(rec['x'])), max(xmax, np.nanmax(rec['x']))
                ymin, ymax = min(ymin, np.nanmin(rec['y'])), max(ymax, np.nanmax(rec['y']))
        span = max(xmax - xmin, ymax - ymin) if np.isfinite(xmin) else 1.0
        span = span if span > 0 else 1.0
        pad = 0.05 * span
        cx, cy = (xmin + xmax) / 2, (ymin + ymax) / 2
        pop_limits[pop] = (cx - span / 2 - pad, cx + span / 2 + pad, cy - span / 2 - pad, cy + span / 2 + pad)

    rows = []
    row_idx = 0
    for pop in FINAL_POP_ORDER:
        pop_color = POP_COLORS[pop]
        x0, x1, y0, y1 = pop_limits[pop]
        for example_idx, uid in enumerate(selected_for_fig[pop]):
            baseline_pop = record_cache[(uid, 'recalculated')]['final_population']
            label_ax = axes[row_idx, 0]
            label_ax.axis('off')
            label_ax.text(
                0.5,
                0.5,
                POP_LABELS[pop] if example_idx == 0 else str(example_idx + 1),
                color=pop_color,
                fontsize=8 if example_idx == 0 else 6,
                fontweight='bold' if example_idx == 0 else 'normal',
                ha='center',
                va='center',
                rotation=90 if example_idx == 0 else 0,
            )

            changed_flags = []
            for col_idx, spec in enumerate(panel_specs, start=1):
                ax = axes[row_idx, col_idx]
                rec = record_cache[(uid, spec['name'])]
                aug_pop = rec['final_population']
                stable = aug_pop == baseline_pop
                changed_flags.append(0 if stable else 1)
                rows.append({
                    'panel': out_stem,
                    'source_population': pop,
                    'baseline_population': baseline_pop,
                    'window_uid': uid,
                    'augmentation': spec['name'],
                    'label': spec['label'].replace('\n', ' '),
                    'aug_population': aug_pop,
                    'stable': stable,
                })
                color = POP_COLORS.get(aug_pop, '#9E9E9E')
                ax.plot(rec['x'], rec['y'], color=color, lw=0.85, alpha=0.95)
                ax.scatter([rec['x'][0]], [rec['y'][0]], c=color, s=6, marker='o', zorder=5)
                ax.scatter([rec['x'][-1]], [rec['y'][-1]], c=color, s=6, marker='s', zorder=5, alpha=0.85)
                if not stable:
                    ax.set_facecolor('#FFEEEE')
                    ax.text(
                        0.03,
                        0.97,
                        f'to {POP_LABELS.get(aug_pop, "Ungated")}',
                        transform=ax.transAxes,
                        ha='left',
                        va='top',
                        fontsize=4.8,
                        color=color,
                        fontweight='bold',
                        bbox=dict(boxstyle='round,pad=0.10', fc='white', ec='none', alpha=0.72),
                    )
                ax.set_xlim(x0, x1)
                ax.set_ylim(y0, y1)
                ax.set_aspect('equal', adjustable='box')
                ax.set_xticks([])
                ax.set_yticks([])
                for spine in ax.spines.values():
                    spine.set_linewidth(0.25)
                    spine.set_color('#BBBBBB')
                if row_idx == 0:
                    ax.set_title(spec['label'], fontsize=6.5, fontweight='bold')

            g1_ax = axes[row_idx, 1 + n_traj_cols]
            draw_gate1_background(g1_ax)
            g2_ax = axes[row_idx, 2 + n_traj_cols]
            draw_gate2_background(g2_ax)
            for spec in panel_specs:
                rec = record_cache[(uid, spec['name'])]
                color = POP_COLORS.get(rec['final_population'], '#9E9E9E')
                g1_ax.scatter(rec['g1_x'], rec['g1_y'], c=color, s=12, ec='white', lw=0.25, alpha=0.88)
                g2_ax.scatter(rec['g2_x'], rec['g2_y'], c=color, s=12, ec='white', lw=0.25, alpha=0.88)
            if row_idx == 0:
                g1_ax.set_title('Gate 1\nmovement', fontsize=6.5, fontweight='bold')
                g2_ax.set_title('Gate 2\nmovement', fontsize=6.5, fontweight='bold')

            hm_ax = axes[row_idx, 3 + n_traj_cols]
            hm_ax.imshow([changed_flags], cmap='RdYlGn_r', vmin=0, vmax=1, aspect='auto')
            hm_ax.set_xticks([])
            hm_ax.set_yticks([])
            if row_idx == 0:
                hm_ax.set_title('Change\nmap', fontsize=6.5, fontweight='bold')
            row_idx += 1

    handles = [plt.Line2D([0], [0], color=POP_COLORS[p], lw=2, label=POP_LABELS[p]) for p in FINAL_POP_ORDER]
    fig.legend(handles=handles, loc='lower center', ncol=len(FINAL_POP_ORDER), frameon=False, bbox_to_anchor=(0.52, -0.003), fontsize=7)
    fig.suptitle(
        panel_title + '\nColor = re-gated pseudo-GT population; pink = changed category; final columns show Gate 1, Gate 2, and binary change map',
        fontsize=9.5,
        fontweight='bold',
        y=0.998,
    )
    plt.subplots_adjust(left=0.015, right=0.995, top=0.945, bottom=0.035, wspace=0.03, hspace=0.03)

    out_svg = OUTPUT_DIR / f'{out_stem}_FINAL.svg'
    out_png = OUTPUT_DIR / f'{out_stem}_FINAL.png'
    fig.savefig(out_svg, format='svg', bbox_inches='tight')
    fig.savefig(out_png, dpi=300, bbox_inches='tight')
    plt.show()

    summary = pd.DataFrame(rows)
    summary_path = OUTPUT_DIR / f'{out_stem}_FINAL_window_summary.csv'
    summary.to_csv(summary_path, index=False)
    print('Saved:', out_svg)
    print('Saved:', out_png)
    print('Saved:', summary_path)
    return summary


shuffle_summary = plot_titration_panel(SHUFFLE_TITRATION, 'A. Shuffle titration', 'augmentation_scan_A_shuffle_titration')
scale_summary = plot_titration_panel(SCALE_TITRATION, 'B. Scale titration', 'augmentation_scan_B_scale_titration')
angle_summary = plot_titration_panel(ANGLE_TITRATION, 'C. Angle titration', 'augmentation_scan_C_angle_titration')

In [ ]:
# Larger-sample stability summary
LARGE_SAMPLE_FRACTION = 0.10
LARGE_MAX_PER_POP = 2000  # set None for the full 10% per population
CHOSEN_SETTING_NAMES = {'shuf_3s_s10_50_a1_5', 'shuf_3s_s10_50_a5_10'}

LARGE_PANEL_SPECS = {
    'Shuffle': SHUFFLE_TITRATION[1:],
    'Scale': SCALE_TITRATION[1:],
    'Angle': ANGLE_TITRATION[1:],
}

def sample_large_complete_uids(pop):
    all_uids = windowed_df.filter(pl.col(SOURCE_LABEL_COL) == pop)['window_uid'].to_list()
    rng.shuffle(all_uids)
    target_n = int(np.ceil(len(all_uids) * LARGE_SAMPLE_FRACTION))
    if LARGE_MAX_PER_POP is not None:
        target_n = min(target_n, int(LARGE_MAX_PER_POP))
    complete = []
    for uid in all_uids:
        if has_complete_frames(instant_df, uid):
            complete.append(uid)
            if len(complete) >= target_n:
                break
    return complete

original_sample_uids = {pop: sample_large_complete_uids(pop) for pop in FINAL_POP_ORDER}
for pop, uids in original_sample_uids.items():
    print(f'{POP_LABELS[pop]:>12} {SOURCE_LABEL_COL} sample: {len(uids):,} complete windows')

# Use the recalculated, unaugmented trajectory category as the baseline/original class.
# This avoids counting changes caused only by recomputing features from raw x/y.
recalc_spec = {'name': 'recalculated', 'label': 'Recalc\ncontrol', 'nseg': None, 'scale': None, 'angle': (0, 0)}
large_sample_uids = {pop: [] for pop in FINAL_POP_ORDER}
recalc_population_by_uid = {}
for original_pop, uids in original_sample_uids.items():
    for uid in uids:
        recalc_rec = augmented_record(uid, recalc_spec)
        if recalc_rec is None:
            continue
        recalc_pop = recalc_rec['final_population']
        recalc_population_by_uid[uid] = recalc_pop
        if recalc_pop in large_sample_uids:
            large_sample_uids[recalc_pop].append(uid)

print('Recalculated-control baseline groups used for change scoring:')
for pop, uids in large_sample_uids.items():
    print(f'{POP_LABELS[pop]:>12}: {len(uids):,} windows')

rows = []
for panel_name, specs in LARGE_PANEL_SPECS.items():
    for spec in specs:
        print(f'Running {panel_name}: {spec["label"].replace(chr(10), " ")}')
        for baseline_pop, uids in large_sample_uids.items():
            for uid in uids:
                rec = augmented_record(uid, spec)
                if rec is None:
                    continue
                rows.append({
                    'panel': panel_name,
                    'population': baseline_pop,
                    'population_label': POP_LABELS[baseline_pop],
                    'baseline_population': baseline_pop,
                    'augmentation': spec['name'],
                    'label': spec['label'].replace('\n', ' '),
                    'aug_population': rec['final_population'],
                    'changed': rec['final_population'] != baseline_pop,
                    'chosen': spec['name'] in CHOSEN_SETTING_NAMES,
                })

large_df = pd.DataFrame(rows)
large_path = OUTPUT_DIR / 'augmentation_scan_large_sample_window_results.csv'
large_df.to_csv(large_path, index=False)
print('Saved:', large_path)

large_summary = (
    large_df.groupby(['panel', 'population_label', 'population', 'augmentation', 'label', 'chosen'], sort=False)
    .agg(n=('changed', 'size'), n_changed=('changed', 'sum'))
    .reset_index()
)
large_summary['pct_changed'] = 100 * large_summary['n_changed'] / large_summary['n']
summary_path = OUTPUT_DIR / 'augmentation_scan_large_sample_summary.csv'
large_summary.to_csv(summary_path, index=False)
display(large_summary)

fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.2), sharey=True)
for ax, panel_name in zip(axes, ['Shuffle', 'Scale', 'Angle']):
    sub = large_summary[large_summary['panel'] == panel_name].copy()
    setting_order = sub[['augmentation', 'label']].drop_duplicates()['augmentation'].tolist()
    x_lookup = {aug: i for i, aug in enumerate(setting_order)}
    for pop in FINAL_POP_ORDER:
        ps = sub[sub['population'] == pop]
        xs = [x_lookup[a] for a in ps['augmentation']]
        ax.plot(xs, ps['pct_changed'], marker='o', lw=1.4, ms=4.5, color=POP_COLORS[pop], label=POP_LABELS[pop])
    for _, row in sub[sub['chosen']].iterrows():
        ax.scatter([x_lookup[row['augmentation']]], [row['pct_changed']], s=130, facecolors='none', edgecolors='black', linewidths=1.3, zorder=10)
    labels = [sub[sub['augmentation'] == aug]['label'].iloc[0] for aug in setting_order]
    ax.set_xticks(range(len(setting_order)))
    ax.set_xticklabels(labels, rotation=35, ha='right', fontsize=8)
    ax.set_title(panel_name, fontweight='bold')
    ax.set_ylabel('% changed category' if panel_name == 'Shuffle' else '')
    ax.grid(axis='y', alpha=0.25)
fig.legend(*axes[0].get_legend_handles_labels(), loc='upper center', ncol=len(FINAL_POP_ORDER), frameon=False, bbox_to_anchor=(0.5, 1.05))
fig.suptitle(
    f'Augmentation stability in larger sampled set ({LARGE_SAMPLE_FRACTION:.0%} per population; cap={LARGE_MAX_PER_POP})\n'
    'Black circles mark selected mild shuffle-scale-angle training setting(s)',
    y=1.17,
    fontsize=11,
    fontweight='bold',
)
plt.tight_layout()
out_svg = OUTPUT_DIR / 'augmentation_scan_large_sample_percent_changed_FINAL.svg'
out_png = OUTPUT_DIR / 'augmentation_scan_large_sample_percent_changed_FINAL.png'
fig.savefig(out_svg, format='svg', bbox_inches='tight')
fig.savefig(out_png, dpi=300, bbox_inches='tight')
plt.show()
print('Saved:', out_svg)
print('Saved:', out_png)
print('Saved:', summary_path)

## Updated Figure Legend

**Supplementary Fig. SX. Offline screen of raw-trajectory augmentation parameters.** Complete 60-frame windows were sampled from the curated pseudo-ground-truth behavioural populations, excluding the mixed exploratory/bound class for this figure. Each row shows one parent trajectory window and each column shows either the recalculated control or an augmented version of the same window. Trajectory color indicates the population assigned after recomputing the gating features and reapplying the hierarchical gate geometry. Pale red backgrounds and the text annotations indicate augmentations that changed category. The final columns show the positions of the recalculated and augmented windows in Gate 1 feature space, Gate 2 feature space, and a binary heatmap of whether category assignment changed. **(A)** Shuffle titration, varying the number of shuffled trajectory segments while leaving scaling and angular perturbation off. **(B)** Scale titration, using the selected 3-segment shuffle while increasing bidirectional per-segment scaling. **(C)** Angle titration, using the selected 3-segment shuffle and 10-50% segment scaling while increasing per-step angular perturbation. **(D)** Larger-sample summary showing the percentage of sampled windows that changed category under each perturbation, split by population; circled points indicate the mild shuffle-scale-angle settings selected for transformer training.